# Fine-tune ViT5

Cấu hình mới: nguồn/đích 1024/256, batch 1 × accumulation 4, AdamW LR 1e-5, AMP FP16, warmup 5% và linear scheduler. Tối đa 30 epoch; giữ validation FP32, early stopping và last/best. Chưa có kết quả chạy cho cấu hình này.

Để tạo run mới, giữ `resume_run_id=""`. Resume chỉ dùng đúng commit và cấu hình của chính run đó. Chạy chuẩn bị dữ liệu trước; đánh giá và demo dùng notebook riêng.


### 1. Môi trường và phiên bản mã

In [ ]:
from pathlib import Path
from google.colab import drive

resume_run_id = ""

drive.mount("/content/drive")
drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

if not drive_root.is_dir():
    raise RuntimeError("Lỗi: Không tìm thấy thư mục dự án trên Drive.")


In [ ]:
import os
import sys
import json
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = ""

if not repo_dir.exists():
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository không đúng dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError("Lỗi: Repository còn thay đổi. Hãy dùng runtime mới.")

subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)

if target_commit == "":
    if resume_run_id != "":
        profile_path = drive_root / "outputs" / resume_run_id / "config.json"
        saved_profile = json.loads(profile_path.read_text(encoding="utf-8"))
        target_commit = saved_profile["source"]["git_commit"]
    else:
        commit_result = subprocess.run(
            ["git", "rev-parse", "origin/main"],
            cwd=repo_dir,
            capture_output=True,
            text=True,
            check=True,
        )
        target_commit = commit_result.stdout.strip()

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit sử dụng:", target_commit)


In [ ]:
%pip uninstall -y -q diffusers torchvision timm fastai
%pip install -q "jedi==0.20.1"
%pip install -q -r /content/vietnamese-summarization/requirements.txt
%pip check


In [ ]:
import gc
import csv
import math
import time
import torch
import src.train
import src.tokenization
from src.utils import load_generation_config, load_train_config, set_seed

config_path = repo_dir / "configs" / "train.json"
train_config = load_train_config(config_path)
generation_config = load_generation_config(
    repo_dir / "configs" / "generation.json"
)

if not torch.cuda.is_available():
    raise RuntimeError("Lỗi: Cần GPU để chạy notebook huấn luyện.")

set_seed(train_config["seed"])

print(json.dumps(
    {
        "seed": train_config["seed"],
        "max_source_length": train_config["max_source_length"],
        "max_target_length": train_config["max_target_length"],
        "batch_size": train_config["batch_size"],
        "gradient_accumulation_steps": train_config["gradient_accumulation_steps"],
        "num_epochs": train_config["num_epochs"],
        "precision": train_config["precision"],
        "learning_rate": train_config["learning_rate"],
        "warmup_ratio": train_config.get("warmup_ratio", 0.0),
        "lr_scheduler_type": train_config.get("lr_scheduler_type", "constant"),
        "generation_config": generation_config,
    },
    ensure_ascii=False,
    indent=2,
))

print("GPU:", torch.cuda.get_device_name(0))


## 2. Dữ liệu, model và optimizer

In [ ]:
tokenized_splits, tokenizer = src.tokenization.tokenize_train_validation(
    drive_root,
    config_path,
)

data_loaders = src.tokenization.create_train_validation_loaders(
    tokenized_splits,
    tokenizer,
    config_path,
)

print("Train:", len(tokenized_splits["train"]))
print("Validation:", len(tokenized_splits["validation"]))


In [ ]:
set_seed(train_config["seed"])

if resume_run_id == "":
    run_profile = src.train.create_training_run(
        repo_dir,
        drive_root,
        allow_uncommitted=False,
    )
    model = src.train.load_vit5_model(config_path)
    optimizer = src.train.create_optimizer(model, config_path)
    scheduler = src.train.create_scheduler(optimizer, data_loaders["train"], config_path)
    scaler = src.train.create_grad_scaler(config_path)
    progress = None
else:
    if Path(resume_run_id).name != resume_run_id or resume_run_id in [".", ".."]:
        raise RuntimeError("Lỗi: Run ID không được chứa đường dẫn.")

    profile_path = drive_root / "outputs" / resume_run_id / "config.json"
    run_profile = json.loads(profile_path.read_text(encoding="utf-8"))

    if run_profile["run_id"] != resume_run_id:
        raise RuntimeError("Lỗi: Hồ sơ không đúng run cần tiếp tục.")

    if run_profile["source"]["git_commit"] != target_commit:
        raise RuntimeError(
            "Lỗi: Resume cần đúng commit của run: "
            + run_profile["source"]["git_commit"]
        )

    model, tokenizer, optimizer, progress, resume_report, scheduler, scaler = (
        src.train.load_last_checkpoint(
            run_profile["paths"]["checkpoint_dir"],
            data_loaders["train"],
            config_path,
            return_training_components=True,
        )
    )
    print("Epoch đã hoàn thành:", progress["completed_epochs"])
    print("Global step:", progress["global_step"])

print("Run huấn luyện:", run_profile["run_id"])
print("Precision:", train_config["precision"], "Master weights:", next(model.parameters()).dtype)
if scheduler is not None:
    print("Total scheduled updates:", scheduler.total_training_steps, "Warmup:", scheduler.warmup_steps)
print("Validation/inference use FP32; training uses AMP when configured.")

output_dir = Path(run_profile["paths"]["output_dir"])
checkpoint_dir = Path(run_profile["paths"]["checkpoint_dir"])
print("Output:", output_dir)
print("Checkpoint:", checkpoint_dir)


## 3. Vòng lặp epoch thực sự huấn luyện


In [ ]:
if run_profile["train_config"] != train_config:
    raise RuntimeError("Lỗi: Cấu hình run không khớp cấu hình hiện tại.")

source_state, source_patch, source_snapshots = src.train.collect_run_source(
    repo_dir,
    allow_uncommitted=False,
)

if source_state["git_commit"] != run_profile["source"]["git_commit"]:
    raise RuntimeError("Lỗi: Phiên bản mã nguồn đã thay đổi.")

if run_profile["source_changes"]["patch_file"] is not None:
    raise RuntimeError("Lỗi: Run dùng bản vá phát triển; cần khôi phục đúng source trước khi resume.")

if run_profile["source_changes"]["untracked_snapshot_files"]:
    raise RuntimeError("Lỗi: Run dùng source snapshot; cần khôi phục đúng source trước khi resume.")

for name in ["last_pending", "last_previous", "best_pending", "best_previous"]:
    if (checkpoint_dir / name).exists():
        raise RuntimeError("Lỗi: Còn checkpoint tạm; cần kiểm tra trước khi huấn luyện.")

if progress is None:
    if (checkpoint_dir / "last").exists() or (checkpoint_dir / "best").exists():
        raise RuntimeError("Lỗi: Run mới không được dùng lại thư mục checkpoint cũ.")
    if len(optimizer.state) != 0 or (output_dir / "training_history.csv").exists():
        raise RuntimeError("Lỗi: Run mới cần optimizer và lịch sử mới.")
    progress = {
        "completed_epochs": 0, "global_step": 0, "training_history": [],
        "attempted_steps": 0, "skipped_steps": 0,
    }

history = list(progress["training_history"])
completed_epochs = progress["completed_epochs"]
updates_per_epoch = (
    len(data_loaders["train"]) + train_config["gradient_accumulation_steps"] - 1
) // train_config["gradient_accumulation_steps"]

if len(history) != completed_epochs or not 0 <= completed_epochs <= train_config["num_epochs"]:
    raise RuntimeError("Lỗi: Lịch sử không khớp số epoch đã hoàn thành.")

# AMP can skip optimizer updates on overflow; consumed groups still count as attempts.
cumulative_updates = 0
cumulative_skips = 0
for index, row in enumerate(history, 1):
    attempts = row.get("attempted_updates", updates_per_epoch)
    skips = row.get("skipped_updates", 0)
    updates = row.get("optimizer_updates", attempts - skips)
    if attempts != updates_per_epoch or not 0 <= skips <= attempts or updates != attempts - skips:
        raise RuntimeError("Lỗi: Bộ đếm cập nhật trong lịch sử không hợp lệ.")
    cumulative_updates += updates
    cumulative_skips += skips
    if row["epoch"] != index or row["global_step"] != cumulative_updates:
        raise RuntimeError("Lỗi: Lịch sử epoch không liên tục.")
if progress["global_step"] != cumulative_updates:
    raise RuntimeError("Lỗi: Global step không khớp các cập nhật thành công.")
if progress.get("attempted_steps", completed_epochs * updates_per_epoch) != completed_epochs * updates_per_epoch:
    raise RuntimeError("Lỗi: Số nhóm đã xử lý không khớp tiến độ.")
if progress.get("skipped_steps", cumulative_skips) != cumulative_skips:
    raise RuntimeError("Lỗi: Số bước AMP bỏ qua không khớp lịch sử.")
if scheduler is not None and scheduler.last_epoch != progress["global_step"]:
    raise RuntimeError("Lỗi: Scheduler không khớp global step.")

validation_ids = []

for sample in tokenized_splits["validation"]:
    validation_ids.append(sample["id"])

train_ids = []

for sample in tokenized_splits["train"]:
    train_ids.append(sample["id"])

if len(train_ids) != run_profile["data"]["train_sample_count"]:
    raise RuntimeError("Lỗi: Số mẫu train không khớp hồ sơ run.")

if len(validation_ids) != run_profile["data"]["validation_sample_count"]:
    raise RuntimeError("Lỗi: Số mẫu validation không khớp hồ sơ run.")

if set(train_ids).intersection(validation_ids):
    raise RuntimeError("Lỗi: Train và validation trùng ID.")

src.train.write_training_history(output_dir, history)
started_at = time.perf_counter()
early_stopping_settings = train_config.get("early_stopping")
early_stopping_state = progress.get("early_stopping")
if early_stopping_settings is not None and early_stopping_state is None:
    # Reconstruct state when resuming a checkpoint without the stored counter.
    for row in history:
        early_stopping_state, _ = src.train.early_stopping(
            early_stopping_state, row["validation_loss"], early_stopping_settings,
        )
stopped_early = (
    early_stopping_settings is not None
    and early_stopping_state is not None
    and early_stopping_state["epochs_without_improvement"] >= early_stopping_settings["patience"]
)

for epoch_index in range(completed_epochs, train_config["num_epochs"]):
    if stopped_early:
        break
    print("Bắt đầu epoch:", epoch_index + 1, flush=True)

    training = src.train.train_epoch(
        model,
        data_loaders["train"],
        optimizer,
        progress["global_step"],
        config_path,
        scheduler=scheduler, scaler=scaler,
    )

    torch.cuda.synchronize()
    validation_started_at = time.perf_counter()
    validation = src.train.validate_model(model, data_loaders["validation"], validation_ids)
    torch.cuda.synchronize()
    validation_seconds = time.perf_counter() - validation_started_at

    if not math.isfinite(training["train_loss"]) or not math.isfinite(validation["validation_loss"]):
        raise RuntimeError("Lỗi: Loss không hữu hạn.")

    progress["completed_epochs"] = epoch_index + 1
    progress["global_step"] = training["global_step"]
    progress["attempted_steps"] = (epoch_index + 1) * updates_per_epoch
    progress["skipped_steps"] = progress.get("skipped_steps", cumulative_skips) + training["skipped_updates"]
    progress, best_decision = src.train.update_best_checkpoint(
        model, tokenizer, checkpoint_dir, progress, validation, config_path,
    )

    history.append({
        "epoch": epoch_index + 1,
        "global_step": training["global_step"],
        "train_loss": training["train_loss"],
        "validation_loss": validation["validation_loss"],
        "train_sample_count": training["train_sample_count"],
        "validation_sample_count": validation["sample_count"],
        "train_valid_target_tokens": training["train_valid_target_tokens"],
        "validation_valid_target_tokens": validation["valid_target_token_count"],
        "learning_rate": optimizer.param_groups[0]["lr"],  # LR for the next successful update
        "optimizer_updates": training["optimizer_updates"],
        "attempted_updates": training["attempted_updates"],
        "skipped_updates": training["skipped_updates"],
        "peak_vram_gib": training["peak_vram_gib"],
        "train_seconds": training["train_seconds"],
        "validation_seconds": validation_seconds,
        "epoch_compute_seconds": training["train_seconds"] + validation_seconds,
    })
    progress["training_history"] = list(history)
    early_stopping_state, stopped_early = src.train.early_stopping(
        early_stopping_state, validation["validation_loss"], early_stopping_settings,
    )
    if early_stopping_settings is not None:
        progress["early_stopping"] = early_stopping_state

    src.train.save_last_checkpoint(
        model, tokenizer, optimizer, checkpoint_dir, progress,
        data_loaders["train"], config_path,
        scheduler=scheduler, scaler=scaler,
    )
    src.train.write_training_history(output_dir, history)

    print(
        "Epoch:", epoch_index + 1,
        "Train loss:", training["train_loss"],
        "Validation loss:", validation["validation_loss"],
        "Cập nhật best:", best_decision["updated"],
        "LR tiếp theo:", optimizer.param_groups[0]["lr"],
        "AMP skipped:", training["skipped_updates"],
        "Peak VRAM GiB:", training["peak_vram_gib"],
        flush=True,
    )

    if stopped_early:
        print(
            "Dừng sớm sau epoch:", epoch_index + 1,
            "Số epoch không cải thiện:", early_stopping_state["epochs_without_improvement"],
            "Best epoch:", progress["best_epoch"],
            flush=True,
        )
        break

training_summary = {
    "run_id": run_profile["run_id"],
    "run_status": "completed",
    "stopped_early": stopped_early,
    "stop_reason": "early_stopping" if stopped_early else "max_epochs",
    "early_stopping": early_stopping_state,
    "inference_checkpoint_dir": str(checkpoint_dir / "best"),
    "completed_epochs": progress["completed_epochs"],
    "global_step": progress["global_step"],
    "history": history,
    "attempted_steps": progress.get("attempted_steps", completed_epochs * updates_per_epoch),
    "skipped_steps": progress.get("skipped_steps", cumulative_skips),
    "training_precision": train_config["precision"],
    "validation_precision": "fp32",
    "best_epoch": progress["best_epoch"],
    "best_global_step": progress["best_global_step"],
    "best_validation_loss": progress["best_validation_loss"],
    "checkpoint_dir": str(checkpoint_dir),
    "history_file": str(output_dir / "training_history.csv"),
    "current_invocation_seconds": time.perf_counter() - started_at,
    "resume_boundary": "end_of_epoch",
    "test_content_read": False,
    "training_entrypoint": "epoch loop in vietnamese_summarization_train.ipynb",
}

summary_path = output_dir / "training_summary.json"
summary_path.write_text(json.dumps(training_summary, ensure_ascii=False, indent=2), encoding="utf-8")

if json.loads(summary_path.read_text(encoding="utf-8")) != training_summary:
    raise RuntimeError("Lỗi: Summary đọc lại không khớp.")

del scheduler
del scaler
del optimizer
del model
gc.collect()
torch.cuda.empty_cache()
print("Hoàn tất run:", training_summary["run_id"])
print("Best epoch:", training_summary["best_epoch"])
print("Best validation loss:", training_summary["best_validation_loss"])


## 4. Loss và phân tích


In [ ]:
report_run_id = run_profile["run_id"]
log_dir = output_dir

summary_path = log_dir / "training_summary.json"
history_path = log_dir / "training_history.csv"

analysis_summary = json.loads(summary_path.read_text(encoding="utf-8"))

if analysis_summary["run_id"] != report_run_id:
    raise RuntimeError("Lỗi: Summary không thuộc run cần phân tích.")

with history_path.open(encoding="utf-8", newline="") as history_file:
    history_rows = list(csv.DictReader(history_file))

if len(history_rows) != len(analysis_summary["history"]):
    raise RuntimeError("Lỗi: CSV và summary khác số epoch.")

for actual, expected in zip(history_rows, analysis_summary["history"]):
    for key in expected:
        if actual[key] != str(expected[key]):
            raise RuntimeError("Lỗi: CSV và summary không khớp tại " + key)

epochs = []
train_losses = []
validation_losses = []

for row in history_rows:
    epochs.append(int(row["epoch"]))
    train_losses.append(float(row["train_loss"]))
    validation_losses.append(float(row["validation_loss"]))

best_epoch = analysis_summary["best_epoch"]
best_loss = analysis_summary["best_validation_loss"]

if best_epoch not in epochs or best_loss != min(validation_losses):
    raise RuntimeError("Lỗi: Thông tin best không khớp lịch sử loss.")

if validation_losses[epochs.index(best_epoch)] != best_loss:
    raise RuntimeError("Lỗi: Best epoch không khớp validation loss.")

print("Run phân tích:", report_run_id)
print("Epoch | Train loss | Validation loss")

for index in range(len(epochs)):
    print(
        epochs[index],
        "|",
        format(train_losses[index], ".6f"),
        "|",
        format(validation_losses[index], ".6f"),
    )

print("Best epoch:", best_epoch)
print("Best validation loss:", best_loss)


In [ ]:
import matplotlib.pyplot as plt

analysis_output_dir = output_dir
figure_dir = analysis_output_dir / "figures"
figure_dir.mkdir(parents=True, exist_ok=True)
plot_path = figure_dir / "training_loss.png"

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(epochs, train_losses, marker="o", label="Train loss")
ax.plot(epochs, validation_losses, marker="o", label="Validation loss")
ax.scatter(
    [best_epoch],
    [best_loss],
    color="seagreen",
    marker="*",
    s=160,
    zorder=5,
    label="Best validation",
)

ax.set_title("ViT5-Base: train and validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Token-averaged cross-entropy loss")
ax.set_xticks(epochs)
ax.set_ylim(0, max(train_losses + validation_losses) * 1.15)
ax.grid(True, alpha=0.3)
ax.legend()

fig.text(0.5, 0.01, report_run_id, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.03, 1, 1])
fig.savefig(plot_path, dpi=200, facecolor="white")
plt.show()

print("Biểu đồ:", plot_path)


In [ ]:
from google.colab import files

analysis_lines = [
    "# Phân tích kết quả huấn luyện",
    "",
    "- Run ID: " + report_run_id,
    "- Nguồn: training_history.csv và training_summary.json của đúng run.",
    "- Best epoch: " + str(best_epoch),
    "- Best global step: " + str(analysis_summary["best_global_step"]),
    "- Best validation loss: " + str(best_loss),
    "- Biểu đồ: [training_loss.png](figures/training_loss.png).",
    "",
    "| Epoch | Train loss | Validation loss |",
    "| --- | --- | --- |",
]

for index in range(len(epochs)):
    analysis_lines.append(
        "| " + str(epochs[index])
        + " | " + format(train_losses[index], ".6f")
        + " | " + format(validation_losses[index], ".6f") + " |"
    )

analysis_lines.append("")

if train_losses[-1] < train_losses[0] and validation_losses[-1] > best_loss:
    analysis_lines.append(
        "Train loss giảm nhưng validation loss ở epoch cuối cao hơn mức tốt nhất. "
        "Đây là dấu hiệu học quá sát train; giữ checkpoint có validation loss nhỏ nhất."
    )
else:
    analysis_lines.append(
        "Chọn checkpoint bằng validation loss nhỏ nhất trong các epoch đã lưu."
    )

analysis_lines.extend([
    "",
    "Loss được tổng hợp theo token đích hợp lệ, bỏ qua nhãn padding -100. "
    "Loss thấp không bảo đảm mọi bản tóm tắt đều đúng hoặc không lặp; "
    "đọc kết quả ROUGE và phân tích lỗi để đánh giá đầy đủ hơn.",
    "",
    "- Số liệu được tạo bởi luồng huấn luyện/resume của chính run này.",
    "- Bảng trên được tính từ log đã lưu, không thay đổi trọng số hoặc dữ liệu test.",
])

analysis_text = "\n".join(analysis_lines) + "\n"
analysis_path = analysis_output_dir / "training_analysis.md"
analysis_path.write_text(analysis_text, encoding="utf-8")

if analysis_path.read_text(encoding="utf-8") != analysis_text:
    raise RuntimeError("Lỗi: File phân tích đọc lại không khớp.")

print(analysis_text)
print("Phân tích:", analysis_path)

import shutil

training_archive = shutil.make_archive(
    str(analysis_output_dir) + "_training_results",
    "zip",
    root_dir=analysis_output_dir,
)
print("Bản sao hồ sơ và kết quả huấn luyện:", training_archive)
files.download(training_archive)
